<a href="https://colab.research.google.com/github/Chimatanagagopal/langchain/blob/main/LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U langgraph langchain langchain-google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.


In [2]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

api_key = userdata.get("GEMINI_API_KEY")

models = [
    "gemini-3.1-flash-lite",
    "gemini-3.5-flash-lite",
    "gemini-3.5-flash",
    "gemini-3.7-flash",
    "gemini-3.6-flash",
]

llms = [
    ChatGoogleGenerativeAI(model=m, google_api_key=api_key, temperature=0, max_retries=2)
    for m in models
]

llm = llms[0].with_fallbacks(llms[1:])

In [3]:
from typing import TypedDict

class InterviewState(TypedDict):
    topic: str
    explanation: str
    question: str

In [4]:
def explanation_node(state: InterviewState):
    topic = state["topic"]

    explanation = llm.invoke(
        f"Explain {topic} in simple English."
    ).content

    return {
        "explanation": explanation
    }

In [5]:
def question_node(state: InterviewState):
    topic = state["topic"]

    question = llm.invoke(
        f"Generate one interview question about {topic}."
    ).content

    return {
        "question": question
    }

In [6]:
from langgraph.graph import StateGraph, START, END
builder = StateGraph(InterviewState)
builder.add_node("explanation", explanation_node)
builder.add_node("question", question_node)
# Connect nodes
builder.add_edge(START, "explanation")
builder.add_edge("explanation", "question")
builder.add_edge("question", END)
# Compile graph
graph = builder.compile()

print("LangGraph compiled successfully!")


LangGraph compiled successfully!


In [7]:
result = graph.invoke({
    "topic": "Python decorators",
    "explanation": "",
    "question": ""
})

print("EXPLANATION:")
print(result["explanation"])

print("\nINTERVIEW QUESTION:")
print(result["question"])

EXPLANATION:
[{'type': 'text', 'text': 'To understand a Python decorator, think of it as a **"wrapper"** or a **"gift-wrapping service"** for your functions.\n\n### The Concept\nImagine you have a function that does a specific job (like making a cup of coffee). Now, imagine you want to add a "feature" to that job (like adding a label that says "Hot!" or cleaning up the counter afterward) without changing the original coffee-making instructions.\n\nA decorator allows you to "wrap" your function with extra code that runs **before** or **after** the function executes.\n\n---\n\n### How it looks in code\nYou identify a decorator by the `@` symbol placed right above a function definition.\n\n```python\n# 1. This is the decorator (the wrapper)\ndef my_decorator(func):\n    def wrapper():\n        print("--- Something happens BEFORE the function ---")\n        func()\n        print("--- Something happens AFTER the function ---")\n    return wrapper\n\n# 2. We apply the decorator using the @ s

In [8]:
!pip install -q langgraph

In [9]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

In [10]:
class InterviewState(TypedDict):
    answer: str
    result: str

In [11]:
def evaluate_answer(state: InterviewState):
    print("Evaluating answer...")

    return {
        "result": "good" if len(state["answer"]) > 20 else "short"
    }


def good_answer(state: InterviewState):
    print("Good answer! Moving to the next question.")

    return {}


def retry_answer(state: InterviewState):
    print("Your answer is too short. Please try again.")

    return {}

In [13]:
def route_answer(state: InterviewState) -> Literal["good_answer", "retry_answer"]:

    if state["result"] == "good":
        return "good_answer"

    return "retry_answer"

In [15]:
builder = StateGraph(InterviewState)

builder.add_node("evaluate_answer", evaluate_answer)
builder.add_node("good_answer", good_answer)
builder.add_node("retry_answer", retry_answer)

builder.add_edge(START, "evaluate_answer")

builder.add_conditional_edges(
    "evaluate_answer",
    route_answer
)

builder.add_edge("good_answer", END)
builder.add_edge("retry_answer", END)

app = builder.compile()

In [16]:

result = app.invoke({
    "answer": "LangGraph helps build stateful AI agent workflows.",
    "result": ""
})

print(result)

Evaluating answer...
Good answer! Moving to the next question.
{'answer': 'LangGraph helps build stateful AI agent workflows.', 'result': 'good'}
